In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    KFold,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)

from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
import joblib


In [2]:
# load from joblib

df_encoded = joblib.load('df_encoded.joblib')
df_encoded

,Price,Bedrooms,Is_premium_subarea,Is_estate,Price_log,Group_median,Group_std,Price_zscore_in_group,Locality_encoded,Area_encoded,Property_type_encoded
0,400000,1,0,1,12.899222,365000.0,1.204600e+06,0.029055,13.818373,14.830809,13.165973
1,14000000,3,1,0,16.454568,7500000.0,5.588542e+06,1.163094,15.667499,14.814729,13.924389
2,200000,1,0,0,12.206078,350000.0,1.978814e+05,-0.758030,13.574728,13.743650,13.143300
3,800000,2,0,0,13.592368,725000.0,3.662361e+05,0.204786,13.705930,13.755032,13.959740
4,300000,1,0,0,12.611541,350000.0,1.978814e+05,-0.252677,13.632339,13.738284,13.121227
...,...,...,...,...,...,...,...,...,...,...,...
8805,2000000,4,0,0,14.508658,1500000.0,3.231787e+05,1.547132,13.776621,13.761360,14.951043
8811,2500000,4,0,0,14.731802,2500000.0,1.000000e+00,0.000000,13.940793,13.743650,15.155985
8812,900000,2,1,0,13.710151,1000000.0,4.179620e+05,-0.239256,13.918081,13.743650,13.961348
8813,3600000,1,0,0,15.096445,400000.0,7.369032e+05,4.342497,14.161802,13.750534,13.141945


In [3]:
y = df_encoded["Price_log"] 
y

0       12.899222
1       16.454568
2       12.206078
3       13.592368
4       12.611541
          ...    
8805    14.508658
8811    14.731802
8812    13.710151
8813    15.096445
8827    15.201805
Name: Price_log, Length: 3196, dtype: float64

In [4]:
X = df_encoded.drop(["Group_median", "Group_std", "Price", "Price_log"], axis=1)
print("""we are not using  column "Group_median", "Group_std", "Price" to train 
because they were gotten from  price directly and price has skewed data
""")
X

we are not using  column "Group_median", "Group_std", "Price" to train 
because they were gotten from  price directly and price has skewed data



,Bedrooms,Is_premium_subarea,Is_estate,Price_zscore_in_group,Locality_encoded,Area_encoded,Property_type_encoded
0,1,0,1,0.029055,13.818373,14.830809,13.165973
1,3,1,0,1.163094,15.667499,14.814729,13.924389
2,1,0,0,-0.758030,13.574728,13.743650,13.143300
3,2,0,0,0.204786,13.705930,13.755032,13.959740
4,1,0,0,-0.252677,13.632339,13.738284,13.121227
...,...,...,...,...,...,...,...
8805,4,0,0,1.547132,13.776621,13.761360,14.951043
8811,4,0,0,0.000000,13.940793,13.743650,15.155985
8812,2,1,0,-0.239256,13.918081,13.743650,13.961348
8813,1,0,0,4.342497,14.161802,13.750534,13.141945


In [5]:
# Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
# Gradient Boosting Setup so that over fitting is prevented from the small data available
gbr = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=3,
    subsample=0.8,
    random_state=42
)

# KFold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Setting up cross_validate()
cv_results = cross_validate(
    estimator=gbr,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring=(
        'neg_root_mean_squared_error',
        'neg_mean_absolute_error',
        'r2'
    ),
    return_train_score=True,
    n_jobs=-1
)
print(cv_results.keys())

# RMSE
cv_rmse = -cv_results['test_neg_root_mean_squared_error']
print("Fold RMSE:", cv_rmse)
print("Mean RMSE:", cv_rmse.mean())

# MAE
cv_mae = -cv_results['test_neg_mean_absolute_error']
print("Fold MAE:", cv_mae)
print("Mean MAE:", cv_mae.mean())

# R2
cv_r2 = cv_results['test_r2']
print("Fold R2:", cv_r2)
print("Mean R2:", cv_r2.mean())

dict_keys(['fit_time', 'score_time', 'test_neg_root_mean_squared_error', 'train_neg_root_mean_squared_error', 'test_neg_mean_absolute_error', 'train_neg_mean_absolute_error', 'test_r2', 'train_r2'])
Fold RMSE: [0.35651308 0.31754272 0.35654928 0.33111448 0.36682833]
Mean RMSE: 0.345709578446722
Fold MAE: [0.22914382 0.20711909 0.23490704 0.22037853 0.22289874]
Mean MAE: 0.22288944708897623
Fold R2: [0.91630841 0.92255438 0.90612464 0.90910591 0.89537142]
Mean R2: 0.9098929523310083


In [14]:
# Detect Overfitting
train_rmse = -cv_results['train_neg_root_mean_squared_error'].mean()
test_rmse = -cv_results['test_neg_root_mean_squared_error'].mean()
print('''If:
train RMSE is much lower than test RMSE
→ model is overfitting.
''')
print(train_rmse)
print(test_rmse)


# fitting the entire dataset to the model that proves best
#gbr.fit(X, y)

# Convert Back to Real Prices Because target is log-transformed:
real_y = np.expm1(y)

If:
train RMSE is much lower than test RMSE
→ model is overfitting.

0.2228571472452101
0.345709578446722


In [24]:
# Cross validate for randomforest

# Random Forest Model
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=15,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# KFold Setup
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# cross_validate()
cv_results = cross_validate(
    estimator=rf,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring=(
        'neg_root_mean_squared_error',
        'neg_mean_absolute_error',
        'r2'
    ),
    return_train_score=True,
    n_jobs=-1
)
print(cv_results.keys())

# RMSE Scores
cv_rmse = -cv_results['test_neg_root_mean_squared_error']
print("Fold RMSE:", cv_rmse)
print("Mean RMSE:", cv_rmse.mean())

#MAE Scores
cv_mae = -cv_results['test_neg_mean_absolute_error']
print("Fold MAE:", cv_mae)
print("Mean MAE:", cv_mae.mean())

# R² Scores
cv_r2 = cv_results['test_r2']
print("Fold R2:", cv_r2)
print("Mean R2:", cv_r2.mean())


dict_keys(['fit_time', 'score_time', 'test_neg_root_mean_squared_error', 'train_neg_root_mean_squared_error', 'test_neg_mean_absolute_error', 'train_neg_mean_absolute_error', 'test_r2', 'train_r2'])
Fold RMSE: [0.3627288  0.33833581 0.40344864 0.35958884 0.3909941 ]
Mean RMSE: 0.3710192386411352
Fold MAE: [0.22927002 0.21906931 0.251063   0.23428013 0.23231908]
Mean MAE: 0.23320030621489746
Fold R2: [0.91336468 0.91207984 0.87980428 0.89280076 0.88113198]
Mean R2: 0.8958363106427999


In [26]:
# Detect Overfitting, Compare train vs validation RMSE.

train_rmse = -cv_results[
    'train_neg_root_mean_squared_error'
].mean()

test_rmse = -cv_results[
    'test_neg_root_mean_squared_error'
].mean()

print("Train RMSE:", train_rmse)
print("Validation RMSE:", test_rmse)

Train RMSE: 0.24326696440358067
Validation RMSE: 0.3710192386411352


In [ ]:
# Train final model
rf.fit(X_train, y_train)

# Convert Back to Real Prices Since target is log-transformed:
real_preds = np.expm1(log_preds)

In [ ]:
# Ridge Regression: Ridge usually performs better than Lasso for housing prices Because linear models require scaling

ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=3.0))
])

ridge_pipeline.fit(X_train, y_train)

ridge_preds = ridge_pipeline.predict(X_test)

In [ ]:
# Lasso Regression

lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=0.001))
])

lasso_pipeline.fit(X_train, y_train)

lasso_preds = lasso_pipeline.predict(X_test)

In [ ]:
# KNN Regressor: KNN absolutely requires scaling.

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsRegressor(
        n_neighbors=8,
        weights='distance',
        p=2
    ))
])

knn_pipeline.fit(X_train, y_train)

knn_preds = knn_pipeline.predict(X_test)

In [ ]:
# GRIDSEARCHCV

from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=model,
    param_grid=params,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)



In [ ]:
import numpy as np
#from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Features
X = df.drop("Price", axis=1)

# Target transformed
y = np.log1p(df["Price"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=53
)

model = RandomForestRegressor()

model.fit(X_train, y_train)

# Predict log price
log_preds = model.predict(X_test)

# Convert back to real price
real_preds = np.expm1(log_preds)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

3
5-fold cross-validation on train set
Evaluate model quality without touching the test set. Use R² and MAE.
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold

model = GradientBoostingRegressor(
    n_estimators=400, max_depth=5,
    learning_rate=0.05, subsample=0.8,
    min_samples_leaf=3, random_state=42)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
r2  = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2')
mae = cross_val_score(model, X_train, y_train, cv=cv,
                      scoring='neg_mean_absolute_error')

print(f"CV R²:  {r2.mean():.3f} ± {r2.std():.3f}")
print(f"CV MAE: {-mae.mean():.3f} ± {mae.std():.3f}  (log scale)")



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Example dataset
df = pd.read_csv("data.csv")

# Separate features and target
X = df.drop("target_column", axis=1)   # all columns except target
y = df["target_column"]                # only target column

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% test data
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


X = df[["age", "salary", "experience"]]
y = df["purchased"]

X = df.drop("purchased", axis=1)
y = df["purchased"]

In [ ]:
WORK FLOW:
1.train/test split
2.scaling where necessary
3.cross validation
4.RMSE + MAE
5.inverse transform using np.expm1